# Workshop : Complexité d'un problème d'optimisation

## Sans complexe

**Compétences visées :**
- [3] Utiliser les notions théoriques de complexité algorithmique
- [3] Étudier la complexité d'un problème de décision
- [2] Estimer la difficulté d'un problème d'optimisation

**Contexte :** Après avoir travaillé sur les cycles eulériens pour la maintenance des luminaires, vous devez maintenant répondre à l'appel de l'ADEME pour optimiser des tournées de livraison. Agathe vous a prévenu : le problème n'est pas le même, et avant de coder quoi que ce soit, il faut déterminer la complexité du problème.

**Objectif du workshop :** À travers une série d'exercices, vous allez :
1. Modéliser le problème de tournée comme un problème d'optimisation connu
2. Formuler le problème de décision associé
3. Prouver son appartenance à NP
4. Comprendre la preuve de NP-Complétude
5. Conclure sur les conséquences pratiques pour votre projet

In [ ]:
# Imports nécessaires pour l'ensemble du workshop
import itertools
import time
import math
import random

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import networkx as nx
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    print("matplotlib et/ou networkx non installés. Certaines visualisations ne seront pas disponibles.")
    print("Installez-les avec : pip install matplotlib networkx")

---

## Partie 1 : Du problème concret au problème formel

### Rappel de la situation

Vous devez optimiser des tournées de livraison pour l'ADEME. Concrètement :
- Vous avez un ensemble de **villes** à livrer
- Vous connaissez les **temps de trajet** entre les villes
- Vous devez partir d'un dépôt, livrer toutes les villes, et **revenir au dépôt**
- L'objectif est de **minimiser le temps total** de la tournée

### Exercice 1.1 -- Identifier le problème

Au prosit précédent, vous avez travaillé sur les **cycles eulériens** (passer par chaque **arête** exactement une fois).

**Question :** En quoi le problème de tournée de livraison est-il fondamentalement différent du problème du cycle eulérien ? Répondez en complétant le tableau ci-dessous.

In [ ]:
# Exercice 1.1 : Complétez le tableau comparatif

tableau = """
| Critère                          | Cycle Eulérien            | Tournée de livraison       |
|----------------------------------|---------------------------|----------------------------|
| On passe par chaque...           | ...                       | ...                        |
| Le graphe doit être...           | ...                       | ...                        |
| C'est un problème de...          | décision / optimisation ? | décision / optimisation ?  |
| Condition d'existence connue ?   | ...                       | ...                        |
"""

print(tableau)
print("Complétez ce tableau sur papier ou dans une cellule markdown ci-dessous.")

<details>
<summary><b>Correction Exercice 1.1</b> (cliquez pour révéler)</summary>

| Critère                          | Cycle Eulérien                    | Tournée de livraison                  |
|----------------------------------|-----------------------------------|---------------------------------------|
| On passe par chaque...           | **arête** exactement une fois     | **sommet** exactement une fois        |
| Le graphe doit être...           | connexe, degrés pairs             | complet (ou complétable)              |
| C'est un problème de...          | **décision** (existe ou non)      | **optimisation** (minimiser le coût)  |
| Condition d'existence connue ?   | Oui (théorème d'Euler)            | Pas de condition simple connue        |

**Point clé :** Passer par chaque **sommet** une seule fois correspond à un **cycle hamiltonien**, pas un cycle eulérien. Ce sont deux problèmes fondamentalement différents !

</details>

### Exercice 1.2 -- Modélisation du problème de tournée

On considère un réseau de villes relié par des routes. Certaines villes ne sont pas directement reliées entre elles.

**Question :** Comment transformer ce graphe incomplet en un graphe complet utilisable pour le problème du Voyageur de Commerce (TSP) ?

**Indication :** Pensez aux plus courts chemins entre paires de sommets non reliés.

In [ ]:
# Exercice 1.2 : Visualisation du problème
# Voici un graphe incomplet représentant un réseau de 5 villes

# Graphe incomplet : matrice de distances (0 = pas de route directe)
graphe_incomplet = {
    ('A', 'B'): 10,
    ('A', 'C'): 25,
    ('B', 'C'): 12,
    ('B', 'D'): 18,
    ('C', 'D'): 8,
    ('C', 'E'): 15,
    ('D', 'E'): 20,
}

villes = ['A', 'B', 'C', 'D', 'E']

print("=== Graphe incomplet (réseau routier) ===")
print("Routes directes :")
for (u, v), d in graphe_incomplet.items():
    print(f"  {u} <-> {v} : {d} km")

print(f"\nPaires sans route directe :")
for i in range(len(villes)):
    for j in range(i+1, len(villes)):
        paire = (villes[i], villes[j])
        paire_inv = (villes[j], villes[i])
        if paire not in graphe_incomplet and paire_inv not in graphe_incomplet:
            print(f"  {villes[i]} <-> {villes[j]} : pas de route directe")

print("\n=> Comment compléter ce graphe pour modéliser le TSP ?")
print("   Écrivez votre réponse sur papier avant de regarder la correction.")

In [ ]:
# Correction Exercice 1.2 : Complétion du graphe par plus courts chemins

import heapq

def dijkstra_complet(villes, aretes):
    """Calcule les plus courts chemins entre toutes les paires de sommets."""
    # Construire la liste d'adjacence
    adj = {v: [] for v in villes}
    for (u, v), d in aretes.items():
        adj[u].append((v, d))
        adj[v].append((u, d))
    
    distances = {}
    for source in villes:
        dist = {v: float('inf') for v in villes}
        dist[source] = 0
        file = [(0, source)]
        while file:
            d, u = heapq.heappop(file)
            if d > dist[u]:
                continue
            for voisin, poids in adj[u]:
                nd = d + poids
                if nd < dist[voisin]:
                    dist[voisin] = nd
                    heapq.heappush(file, (nd, voisin))
        for v in villes:
            if source != v:
                cle = (min(source, v), max(source, v))
                if cle not in distances or dist[v] < distances[cle]:
                    distances[cle] = dist[v]
    return distances

graphe_complet = dijkstra_complet(villes, graphe_incomplet)

print("=== Graphe complet (après calcul des plus courts chemins) ===")
print(f"{'Paire':>10} {'Distance directe':>18} {'Plus court chemin':>20}")
print("-" * 52)
for i in range(len(villes)):
    for j in range(i+1, len(villes)):
        paire = (villes[i], villes[j])
        direct = graphe_incomplet.get(paire, graphe_incomplet.get((villes[j], villes[i]), None))
        pcc = graphe_complet[paire]
        directe_str = str(direct) if direct else "pas de route"
        print(f"{villes[i]}-{villes[j]:>8} {directe_str:>18} {pcc:>20}")

print("\n=> Les distances du graphe complet respectent-elles l'inégalité triangulaire ?")
print("   Pour tous sommets i, j, k : d(i,j) <= d(i,k) + d(k,j)")
print("   C'est le cas car on utilise les plus courts chemins !")
print("   => On obtient la version MÉTRIQUE du TSP (Delta-TSP)")

<details>
<summary><b>Correction Exercice 1.2 -- Synthèse</b> (cliquez pour révéler)</summary>

**Méthode de complétion :**
1. On conserve les arêtes existantes avec leurs poids
2. Pour chaque paire de sommets non reliés, on calcule le **plus court chemin** dans le graphe original (ex: Dijkstra)
3. On ajoute une arête avec ce poids

**Propriété importante :** Le graphe complet ainsi construit respecte l'**inégalité triangulaire** :
$$\forall i, j, k : d(i,j) \leq d(i,k) + d(k,j)$$

Car les poids sont des plus courts chemins. On se ramène donc à la **version métrique du TSP** ($\Delta$-TSP).

**Conséquence pratique :** Un cycle hamiltonien dans ce graphe complet peut emprunter plusieurs fois la même route physique (en passant par des villes intermédiaires), ce qui ne remet pas en cause l'optimalité.

</details>

---

## Partie 2 : Problème de décision vs problème d'optimisation

Agathe insiste : avant de chercher un algorithme, il faut d'abord **formuler le problème de décision** associé au problème d'optimisation, car c'est sur les problèmes de décision que la théorie de la complexité s'appuie.

### Rappel : lien entre décision et optimisation

- **Problème d'optimisation :** Trouver la meilleure solution (ex : le cycle hamiltonien le plus court)
- **Problème de décision associé :** Existe-t-il une solution respectant une borne k ? (ex : existe-t-il un cycle hamiltonien de longueur $\leq k$ ?)

Résoudre le problème d'optimisation revient à trouver la plus petite valeur de $k$ pour laquelle la réponse au problème de décision est "oui".

### Exercice 2.1 -- Formuler le problème de décision

**Question :** Formulez le problème de décision associé au TSP métrique. Précisez :
1. Les données d'entrée (instance du problème)
2. La question posée

In [ ]:
# Exercice 2.1 : Illustration concrète de la différence décision / optimisation
# Utilisons le graphe complet calculé précédemment

def distance_graphe_complet(u, v, distances):
    """Retourne la distance entre u et v dans le graphe complet."""
    cle = (min(u, v), max(u, v))
    return distances.get(cle, float('inf'))

def cout_cycle(cycle, distances):
    """Calcule le coût total d'un cycle."""
    total = 0
    for i in range(len(cycle)):
        total += distance_graphe_complet(cycle[i], cycle[(i+1) % len(cycle)], distances)
    return total

# Énumération exhaustive de tous les cycles possibles
depart = 'A'
autres = [v for v in villes if v != depart]
meilleur_cout = float('inf')
meilleur_cycle = None

print("=== Tous les cycles hamiltoniens possibles ===\n")
for perm in itertools.permutations(autres):
    cycle = [depart] + list(perm)
    c = cout_cycle(cycle, graphe_complet)
    marqueur = ""
    if c < meilleur_cout:
        meilleur_cout = c
        meilleur_cycle = cycle
        marqueur = " <-- nouveau meilleur !"
    print(f"  {' -> '.join(cycle)} -> {depart} : {c} km{marqueur}")

print(f"\n=== Problème d'OPTIMISATION ===")
print(f"Meilleur cycle : {' -> '.join(meilleur_cycle)} -> {depart}")
print(f"Coût optimal   : {meilleur_cout} km")

print(f"\n=== Problème de DÉCISION ===")
for k in [60, 70, 80, 90, 100]:
    reponse = "OUI" if meilleur_cout <= k else "NON"
    print(f"  Existe-t-il un cycle de coût <= {k} km ? {reponse}")

<details>
<summary><b>Correction Exercice 2.1</b> (cliquez pour révéler)</summary>

**Problème de décision du TSP métrique :**

| Élément | Description |
|---------|-------------|
| **Données** | $G = (V, E, P)$ un graphe complet arête-pondéré respectant l'inégalité triangulaire, un sommet $v \in V$, un entier $k$ |
| **Question** | Existe-t-il un cycle hamiltonien partant de $v$, de longueur $\leq k$ ? |

**Problème d'optimisation associé :** Quelle est la plus petite valeur de $k$ pour laquelle la réponse est "oui" ?

**Observation importante :** Le nombre de cycles hamiltoniens possibles est $(n-1)!$ . Pour $n = 20$ villes, cela fait déjà $\approx 1.2 \times 10^{17}$ cycles à tester ! On ne peut pas tous les énumérer.

</details>

### Exercice 2.2 -- Explosion combinatoire

**Question :** Calculez le nombre de cycles hamiltoniens possibles pour différentes valeurs de $n$. À partir de quelle taille l'énumération exhaustive devient-elle impraticable ?

In [ ]:
# Exercice 2.2 : Explosion combinatoire
# Complétez la colonne "temps estimé" en supposant 10^9 opérations/seconde

ops_par_seconde = 1e9

print(f"{'n villes':>10} {'(n-1)! cycles':>20} {'Temps estimé':>25}")
print("-" * 58)

for n in [5, 10, 12, 15, 20, 25, 30, 50]:
    nb_cycles = math.factorial(n - 1)
    temps_sec = nb_cycles / ops_par_seconde
    
    if temps_sec < 1:
        temps_str = f"{temps_sec*1000:.4f} ms"
    elif temps_sec < 60:
        temps_str = f"{temps_sec:.2f} secondes"
    elif temps_sec < 3600:
        temps_str = f"{temps_sec/60:.1f} minutes"
    elif temps_sec < 86400:
        temps_str = f"{temps_sec/3600:.1f} heures"
    elif temps_sec < 365.25 * 86400:
        temps_str = f"{temps_sec/86400:.1f} jours"
    elif temps_sec < 365.25 * 86400 * 1e6:
        temps_str = f"{temps_sec/(365.25*86400):.1f} années"
    else:
        temps_str = f"{temps_sec/(365.25*86400):.2e} années"
    
    print(f"{n:>10} {nb_cycles:>20} {temps_str:>25}")

print("\n=> À partir de quelle valeur de n l'approche exhaustive est-elle inutilisable ?")
print("   Quelle conclusion en tirez-vous sur la nécessité d'étudier la complexité ?")

<details>
<summary><b>Correction Exercice 2.2</b> (cliquez pour révéler)</summary>

- Pour **n = 12**, l'énumération prend déjà plusieurs minutes
- Pour **n = 15**, on dépasse les jours de calcul
- Pour **n = 20**, on atteint des milliers d'années !

**Conclusion :** L'approche exhaustive (force brute) est en $O(n!)$, ce qui est **bien pire que l'exponentiel** $O(2^n)$. Même avec un supercalculateur, on ne peut pas énumérer toutes les solutions pour des instances de taille réaliste.

La question cruciale est donc : **peut-on faire mieux ?** C'est exactement la question à laquelle la théorie de la complexité permet de répondre.

</details>

---

## Partie 3 : Le problème de décision est-il dans NP ?

Agathe vous dit : *"Commence d'abord par voir si le problème de décision associé est dans NP. Imagine si une Machine de Turing n'est même pas capable de décider de la validité d'une solution..."*

### Rappel : qu'est-ce que la classe NP ?

Un problème de décision est dans **NP** s'il existe un algorithme (appelé **certificat** ou **vérificateur**) qui :
1. Prend en entrée une **instance** du problème et une **solution candidate**
2. Vérifie en **temps polynomial** si cette solution candidate répond "oui" à l'instance

**Attention :** NP signifie "Non-déterministe Polynomial", et **non** "Non Polynomial" !

### Exercice 3.1 -- Écrire un algorithme de certificat pour le TSP

**Question :** Écrivez un algorithme (en pseudo-code ou en Python) qui, étant donné :
- Un graphe complet pondéré $G = (V, E, P)$
- Un sommet de départ $v$
- Une borne $k$
- Une **solution candidate** (une suite de sommets)

Vérifie en **temps polynomial** si la solution candidate est une réponse valide "oui" au problème de décision.

In [ ]:
# Exercice 3.1 : Complétez l'algorithme de certificat

def certificat_tsp(sommets, distances, depart, k, solution_candidate):
    """
    Vérifie si solution_candidate est un cycle hamiltonien
    de longueur <= k partant de 'depart'.
    
    Paramètres :
        sommets : liste des sommets du graphe
        distances : dict {(u,v): poids} pour le graphe complet
        depart : sommet de départ
        k : borne maximale
        solution_candidate : liste de sommets (le cycle proposé)
    
    Retourne : True si la solution est valide, False sinon
    """
    n = len(sommets)
    
    # TODO 1 : Vérifier que la solution candidate a le bon nombre de sommets
    # ...
    
    # TODO 2 : Vérifier que le cycle part bien du sommet de départ
    # ...
    
    # TODO 3 : Vérifier que c'est un cycle hamiltonien
    #          (chaque sommet apparaît exactement une fois)
    # ...
    
    # TODO 4 : Calculer la longueur totale du cycle et vérifier qu'elle est <= k
    # ...
    
    pass  # Remplacez par votre implémentation


# Test de votre certificat
print("=== Test de l'algorithme de certificat ===\n")

# Solution valide
sol1 = ['A', 'B', 'D', 'E', 'C']
print(f"Solution 1 : {sol1}")
print(f"  Résultat attendu : True (cycle hamiltonien de coût <= 100)")
# print(f"  Votre résultat   : {certificat_tsp(villes, graphe_complet, 'A', 100, sol1)}")

# Solution invalide : sommet manquant
sol2 = ['A', 'B', 'C', 'D']
print(f"\nSolution 2 : {sol2}")
print(f"  Résultat attendu : False (il manque le sommet E)")
# print(f"  Votre résultat   : {certificat_tsp(villes, graphe_complet, 'A', 100, sol2)}")

# Solution invalide : ne part pas du bon sommet
sol3 = ['B', 'A', 'C', 'D', 'E']
print(f"\nSolution 3 : {sol3}")
print(f"  Résultat attendu : False (ne part pas de A)")
# print(f"  Votre résultat   : {certificat_tsp(villes, graphe_complet, 'A', 100, sol3)}")

print("\n=> Décommentez les lignes une fois votre implémentation terminée.")

In [ ]:
# Correction Exercice 3.1 : Algorithme de certificat

def certificat_tsp_corrige(sommets, distances, depart, k, solution_candidate):
    """
    Certificat polynomial pour le TSP.
    Vérifie si solution_candidate est un cycle hamiltonien de longueur <= k.
    """
    n = len(sommets)
    
    # Étape 1 : Vérifier que la solution a le bon nombre de sommets -> O(1)
    if len(solution_candidate) != n:
        return False, "Nombre de sommets incorrect"
    
    # Étape 2 : Vérifier que le cycle part du bon sommet -> O(1)
    if solution_candidate[0] != depart:
        return False, f"Le cycle ne part pas de {depart}"
    
    # Étape 3 : Vérifier que c'est un cycle hamiltonien -> O(n)
    # Chaque sommet doit apparaître exactement une fois
    sommets_vus = set()
    for s in solution_candidate:
        if s in sommets_vus:
            return False, f"Le sommet {s} apparaît plus d'une fois"
        if s not in sommets:
            return False, f"Le sommet {s} n'existe pas dans le graphe"
        sommets_vus.add(s)
    
    if sommets_vus != set(sommets):
        manquants = set(sommets) - sommets_vus
        return False, f"Sommets manquants : {manquants}"
    
    # Étape 4 : Calculer la longueur et vérifier <= k -> O(n)
    longueur = 0
    for i in range(n):
        u = solution_candidate[i]
        v = solution_candidate[(i + 1) % n]
        cle = (min(u, v), max(u, v))
        longueur += distances[cle]
    
    if longueur > k:
        return False, f"Longueur {longueur} > borne {k}"
    
    return True, f"Cycle hamiltonien valide de longueur {longueur} <= {k}"


# Tests
print("=== Test du certificat corrigé ===\n")

tests = [
    (['A', 'B', 'D', 'E', 'C'], 100, "cycle valide, borne large"),
    (['A', 'B', 'D', 'E', 'C'], 50, "cycle valide, borne trop stricte"),
    (['A', 'B', 'C', 'D'], 100, "trop peu de sommets"),
    (['B', 'A', 'C', 'D', 'E'], 100, "mauvais départ"),
    (['A', 'B', 'B', 'D', 'E'], 100, "sommet répété"),
]

for sol, k, description in tests:
    valide, msg = certificat_tsp_corrige(villes, graphe_complet, 'A', k, sol)
    status = "VALIDE" if valide else "INVALIDE"
    print(f"  [{status:>8}] {sol} (k={k}) -- {description}")
    print(f"            -> {msg}\n")

print("=== Analyse de la complexité du certificat ===")
print("  Étape 1 : O(1)")
print("  Étape 2 : O(1)")
print("  Étape 3 : O(n) -- parcours de tous les sommets")
print("  Étape 4 : O(n) -- parcours de toutes les arêtes du cycle")
print("  -------")
print("  TOTAL   : O(n) -- POLYNOMIAL !")
print("\n=> Le problème de décision du TSP est donc dans NP.")

### Exercice 3.2 -- Certificat pour le problème de la n-coloration

Pour vérifier que vous avez bien compris la notion de certificat, appliquons-la à un autre problème.

**Problème de la n-coloration de graphe :**
> Étant donné un graphe $G = (V, E)$ et un entier $n$, peut-on colorier ses sommets avec au plus $n$ couleurs, de manière à ce qu'aucune paire de sommets voisins n'ait la même couleur ?

**Question :** Écrivez un algorithme de certificat pour ce problème. Quelle est sa complexité ?

In [ ]:
# Exercice 3.2 : Complétez le certificat pour la n-coloration

def certificat_coloration(sommets, aretes, n, coloration_candidate):
    """
    Vérifie si coloration_candidate est une coloration valide
    utilisant au plus n couleurs.
    
    Paramètres :
        sommets : liste des sommets
        aretes : liste de tuples (u, v)
        n : nombre maximal de couleurs autorisées
        coloration_candidate : dict {sommet: couleur}
    
    Retourne : True si la coloration est valide, False sinon
    """
    # TODO : Implémentez le certificat
    # 1. Vérifier que chaque sommet a une couleur
    # 2. Vérifier que le nombre de couleurs utilisées est <= n
    # 3. Vérifier qu'aucune paire de sommets voisins n'a la même couleur
    
    pass  # Remplacez par votre implémentation


# Test
sommets_g = [1, 2, 3, 4, 5]
aretes_g = [(1, 2), (1, 3), (2, 3), (3, 4), (4, 5)]

# Coloration valide avec 3 couleurs
col1 = {1: 'R', 2: 'B', 3: 'V', 4: 'R', 5: 'B'}
print(f"Coloration 1 : {col1}")
print(f"  Attendu : True (3-coloration valide)")
# print(f"  Résultat : {certificat_coloration(sommets_g, aretes_g, 3, col1)}")

# Coloration invalide (voisins de même couleur)
col2 = {1: 'R', 2: 'R', 3: 'V', 4: 'R', 5: 'B'}
print(f"\nColoration 2 : {col2}")
print(f"  Attendu : False (sommets 1 et 2 sont voisins et de même couleur)")
# print(f"  Résultat : {certificat_coloration(sommets_g, aretes_g, 3, col2)}")

<details>
<summary><b>Correction Exercice 3.2</b> (cliquez pour révéler)</summary>

```python
def certificat_coloration(sommets, aretes, n, coloration_candidate):
    # 1. Vérifier que chaque sommet a une couleur -> O(|V|)
    for s in sommets:
        if s not in coloration_candidate:
            return False
    
    # 2. Compter les couleurs utilisées -> O(|V|)
    couleurs = set(coloration_candidate.values())
    if len(couleurs) > n:
        return False
    
    # 3. Vérifier la contrainte de voisinage -> O(|E|)
    for u, v in aretes:
        if coloration_candidate[u] == coloration_candidate[v]:
            return False
    
    return True
```

**Complexité :** $O(|V| + |E|) \leq O(n^2)$ car $|E| \leq \frac{n(n-1)}{2}$

Le problème de la n-coloration est donc aussi **dans NP**.

</details>

---

## Partie 4 : Preuve de NP-Complétude du TSP

On a montré que le TSP est **dans NP** (on sait vérifier une solution en temps polynomial). Il reste à montrer qu'il est **NP-Difficile**, c'est-à-dire au moins aussi difficile que tout problème de NP.

Pour cela, on utilise une **réduction polynomiale** : on montre qu'un problème déjà connu comme NP-Complet peut se transformer en une instance du TSP.

### Rappel : principe de la réduction polynomiale

Pour prouver que le problème B est NP-Complet, on prend un problème A **déjà connu** comme NP-Complet et on montre que :
1. Toute instance de A peut être transformée en instance de B **en temps polynomial**
2. La réponse est conservée : A répond "oui" $\Leftrightarrow$ B répond "oui"

On note $A \leq_p B$ ("A se réduit polynomialement à B"), ce qui signifie que **B est au moins aussi difficile que A**.

### Exercice 4.1 -- Comprendre le sens de la réduction

**Question :** Pour prouver que le TSP est NP-Complet en utilisant le cycle hamiltonien (connu NP-Complet depuis 1972), dans quel sens doit-on faire la réduction ?

- (a) Transformer une instance du TSP en instance du cycle hamiltonien ?
- (b) Transformer une instance du cycle hamiltonien en instance du TSP ?

<details>
<summary><b>Correction Exercice 4.1</b> (cliquez pour révéler)</summary>

La réponse est **(b)** : on doit transformer une instance du **cycle hamiltonien** en une instance du **TSP**.

**Raisonnement :** On veut montrer que le TSP est **au moins aussi difficile** que le cycle hamiltonien. Si on sait résoudre le TSP, alors on peut aussi résoudre le cycle hamiltonien (en transformant l'instance). Donc le TSP est au moins aussi dur.

$\text{Cycle Hamiltonien} \leq_p \text{TSP}$

**Erreur fréquente :** réduire dans le mauvais sens ! Réduire le TSP vers le cycle hamiltonien prouverait que le cycle hamiltonien est au moins aussi difficile que le TSP, ce qui n'est pas ce qu'on veut.

</details>

### Exercice 4.2 -- La réduction Cycle Hamiltonien $\rightarrow$ TSP

Le problème du **cycle hamiltonien** est le suivant :
> Étant donné un graphe $G = (V, E)$ non pondéré, existe-t-il un cycle passant par chaque sommet exactement une fois ?

Ce problème est **NP-Complet** (prouvé par Karp en 1972).

**Question :** Proposez une méthode pour transformer une instance du cycle hamiltonien (un graphe non pondéré) en une instance du TSP (un graphe complet pondéré + une borne k). L'objectif est que les deux instances aient la même réponse.

In [ ]:
# Exercice 4.2 : Implémentez la réduction Cycle Hamiltonien -> TSP

def reduction_hamiltonien_vers_tsp(sommets, aretes):
    """
    Transforme une instance du problème du cycle hamiltonien
    en une instance du problème du TSP.
    
    Paramètres :
        sommets : liste des sommets du graphe original
        aretes : ensemble de tuples (u, v) représentant les arêtes
    
    Retourne :
        distances : dict des distances du graphe complet pondéré
        k : borne pour le problème de décision du TSP
    """
    n = len(sommets)
    aretes_norm = set()
    for u, v in aretes:
        aretes_norm.add((min(u, v), max(u, v)))
    
    # TODO : Construire le graphe complet pondéré G'
    # - Si l'arête existe dans G : poids = ???
    # - Si l'arête n'existe pas dans G : poids = ???
    # TODO : Choisir la borne k
    
    distances = {}
    # Votre code ici...
    
    k = 0  # Votre borne ici...
    
    return distances, k


# Graphe de test AVEC cycle hamiltonien : 1-2-3-4-5-1
sommets_test = [1, 2, 3, 4, 5]
aretes_test = {(1, 2), (2, 3), (3, 4), (4, 5), (5, 1), (1, 3)}

print("=== Graphe original G ===")
print(f"Sommets : {sommets_test}")
print(f"Arêtes  : {aretes_test}")
print(f"Cycle hamiltonien : 1-2-3-4-5-1 existe !")

print("\n=> Complétez la fonction de réduction ci-dessus")
print("   puis décommentez les tests ci-dessous")

# dist, borne = reduction_hamiltonien_vers_tsp(sommets_test, aretes_test)
# print(f"\n=== Instance TSP (après réduction) ===")
# print(f"Distances : {dist}")
# print(f"Borne k   : {borne}")

In [ ]:
# Correction Exercice 4.2 : Réduction Cycle Hamiltonien -> TSP

def reduction_hamiltonien_vers_tsp_corrige(sommets, aretes):
    """
    Réduction polynomiale du cycle hamiltonien vers le TSP.
    
    Construction :
    - G' = graphe complet sur les mêmes sommets
    - Poids 1 si l'arête existe dans G, poids 2 sinon
    - Borne k = |V| (nombre de sommets)
    """
    n = len(sommets)
    aretes_norm = set()
    for u, v in aretes:
        aretes_norm.add((min(u, v), max(u, v)))
    
    # Construire le graphe complet pondéré
    distances = {}
    for i in range(len(sommets)):
        for j in range(i + 1, len(sommets)):
            u, v = sommets[i], sommets[j]
            cle = (min(u, v), max(u, v))
            # Poids 1 si arête dans G, poids 2 sinon
            distances[cle] = 1 if cle in aretes_norm else 2
    
    # Borne k = n (nombre de sommets)
    k = n
    
    return distances, k


# === Démonstration de la réduction ===

# Cas 1 : Graphe AVEC cycle hamiltonien
print("=" * 60)
print("CAS 1 : Graphe avec cycle hamiltonien")
print("=" * 60)

sommets_1 = [1, 2, 3, 4, 5]
aretes_1 = {(1, 2), (2, 3), (3, 4), (4, 5), (5, 1), (1, 3)}

dist_1, k_1 = reduction_hamiltonien_vers_tsp_corrige(sommets_1, aretes_1)

print(f"Graphe G : sommets={sommets_1}, arêtes={aretes_1}")
print(f"Instance TSP : k = {k_1}")
print(f"Distances : {dist_1}")

# Vérifier avec le cycle 1-2-3-4-5-1 (toutes arêtes de poids 1)
cycle_1 = [1, 2, 3, 4, 5]
cout_1 = sum(dist_1[(min(cycle_1[i], cycle_1[(i+1)%5]), max(cycle_1[i], cycle_1[(i+1)%5]))] for i in range(5))
print(f"\nCycle 1-2-3-4-5-1 : coût = {cout_1}")
print(f"Coût <= k ({k_1}) ? {'OUI' if cout_1 <= k_1 else 'NON'}")
print(f"=> G a un cycle hamiltonien ET le TSP répond OUI")

# Cas 2 : Graphe SANS cycle hamiltonien
print(f"\n{'=' * 60}")
print("CAS 2 : Graphe sans cycle hamiltonien")
print("=" * 60)

sommets_2 = [1, 2, 3, 4, 5]
aretes_2 = {(1, 2), (2, 3)}  # Pas assez d'arêtes

dist_2, k_2 = reduction_hamiltonien_vers_tsp_corrige(sommets_2, aretes_2)

print(f"Graphe G : sommets={sommets_2}, arêtes={aretes_2}")
print(f"Instance TSP : k = {k_2}")

# Tester tous les cycles possibles
meilleur = float('inf')
for perm in itertools.permutations(sommets_2[1:]):
    cycle = [sommets_2[0]] + list(perm)
    cout = sum(dist_2[(min(cycle[i], cycle[(i+1)%5]), max(cycle[i], cycle[(i+1)%5]))] for i in range(5))
    if cout < meilleur:
        meilleur = cout

print(f"Meilleur cycle possible : coût = {meilleur}")
print(f"Coût <= k ({k_2}) ? {'OUI' if meilleur <= k_2 else 'NON'}")
print(f"=> G n'a PAS de cycle hamiltonien ET le TSP répond NON")

print(f"\n{'=' * 60}")
print("CONCLUSION")
print("=" * 60)
print("La réduction conserve la réponse dans les deux sens :")
print("  G a un cycle hamiltonien <=> TSP(G', k=n) répond OUI")
print("  La transformation est en O(n²) => polynomiale")
print("  => Le TSP est NP-Complet !")

### Exercice 4.3 -- Le cas métrique (Delta-TSP)

Agathe fait une remarque importante : notre problème de tournée n'est pas le TSP général, mais le **TSP métrique** (les distances respectent l'inégalité triangulaire).

**Question :** La preuve de NP-Complétude du TSP général s'applique-t-elle aussi au TSP métrique ? Justifiez en examinant les poids utilisés dans la réduction.

**Indication :** Vérifiez si les distances 1 et 2 utilisées dans la réduction respectent l'inégalité triangulaire.

In [ ]:
# Exercice 4.3 : Vérification de l'inégalité triangulaire

def verifier_inegalite_triangulaire(sommets, distances):
    """Vérifie si les distances respectent l'inégalité triangulaire."""
    violations = []
    n = len(sommets)
    
    for i in range(n):
        for j in range(n):
            for k in range(n):
                if i != j and j != k and i != k:
                    si, sj, sk = sommets[i], sommets[j], sommets[k]
                    d_ij = distances.get((min(si, sj), max(si, sj)), float('inf'))
                    d_ik = distances.get((min(si, sk), max(si, sk)), float('inf'))
                    d_kj = distances.get((min(sk, sj), max(sk, sj)), float('inf'))
                    
                    if d_ij > d_ik + d_kj:
                        violations.append((si, sj, sk, d_ij, d_ik, d_kj))
    
    return violations

# Vérifions sur l'instance TSP construite par la réduction
print("=== Vérification de l'inégalité triangulaire ===")
print("Pour les distances construites par la réduction (poids 1 et 2) :\n")

violations = verifier_inegalite_triangulaire(sommets_1, dist_1)

if not violations:
    print("  Aucune violation ! L'inégalité triangulaire est respectée.")
    print("\n  En effet, les poids sont 1 ou 2, donc :")
    print("  - d(i,j) <= 2 (poids max)")
    print("  - d(i,k) + d(k,j) >= 1 + 1 = 2 (somme minimale)")
    print("  => d(i,j) <= d(i,k) + d(k,j) est toujours vrai !")
else:
    print(f"  {len(violations)} violation(s) trouvée(s) !")
    for v in violations[:5]:
        print(f"  d({v[0]},{v[1]}) = {v[3]} > d({v[0]},{v[2]}) + d({v[2]},{v[1]}) = {v[4]} + {v[5]}")

print("\n=> La réduction construit des instances qui sont AUSSI des instances")
print("   du TSP métrique. Donc le Delta-TSP est également NP-Complet.")

<details>
<summary><b>Correction Exercice 4.3 -- Résumé de la preuve</b> (cliquez pour révéler)</summary>

**Oui, la preuve s'applique aussi au TSP métrique.** Voici pourquoi :

La réduction utilise des poids **1** (arête présente) et **2** (arête absente). Pour tous sommets $i$, $j$, $k$ :
- $d(i,j) \leq 2$ (poids maximal)
- $d(i,k) + d(k,j) \geq 1 + 1 = 2$ (somme minimale)

Donc $d(i,j) \leq d(i,k) + d(k,j)$ est **toujours vérifié**. L'inégalité triangulaire est respectée.

Les instances construites par la réduction sont donc des instances valides du $\Delta$-TSP. Un algorithme polynomial pour le $\Delta$-TSP résoudrait aussi le cycle hamiltonien, ce qui est impossible (sauf si $P = NP$).

**Remarque fondamentale :** Un cas particulier d'un problème NP-Complet peut parfois être polynomial. Ici, ce n'est **pas** le cas, mais il est essentiel de toujours vérifier !

</details>

---

## Partie 5 : Conséquences sur le problème d'optimisation

Nous avons maintenant prouvé que le TSP (et sa version métrique) est **NP-Complet**. Qu'est-ce que cela implique concrètement pour notre projet de tournées de livraison ?

### Exercice 5.1 -- Interpréter la NP-Complétude

**Question :** Parmi les affirmations suivantes, lesquelles sont **vraies** ? Justifiez chacune de vos réponses.

1. "Le TSP est insoluble, on ne peut jamais trouver de solution."
2. "On ne peut pas trouver d'algorithme polynomial qui garantit la solution optimale."
3. "Il n'existe aucun algorithme pour résoudre le TSP."
4. "Un algorithme de force brute peut résoudre le TSP pour de petites instances."
5. "Si on prouve un jour que P=NP, le TSP deviendra facile à résoudre en pratique."
6. "La NP-Complétude prouve qu'il faut choisir entre optimalité et temps raisonnable."

<details>
<summary><b>Correction Exercice 5.1</b> (cliquez pour révéler)</summary>

1. **FAUX** -- Le TSP n'est pas insoluble. On peut toujours trouver une solution (même par force brute). C'est juste que le temps de calcul peut être astronomique pour de grandes instances.

2. **VRAI** -- C'est exactement ce que signifie la NP-Complétude (sauf si P=NP, ce qui est très improbable). On ne peut pas avoir **à la fois** la garantie d'optimalité **et** un temps polynomial.

3. **FAUX** -- La force brute en $O(n!)$ résout le TSP. L'algorithme de Held-Karp le résout en $O(n^2 \cdot 2^n)$. Il existe des algorithmes, mais ils sont tous de complexité exponentielle (ou pire).

4. **VRAI** -- Pour $n \leq 15$, la force brute reste praticable en quelques secondes.

5. **PARTIELLEMENT VRAI** -- Si P=NP, il **existerait** un algorithme polynomial. Mais si le polynôme est de degré élevé (ex : $n^{1000}$) avec une grosse constante, la résolution resterait impraticable en pratique.

6. **VRAI** -- C'est la conclusion fondamentale :
   - Soit on cherche l'**optimum** avec un algorithme **exponentiel** (petites instances uniquement)
   - Soit on accepte une solution **sous-optimale** avec un algorithme **polynomial** (heuristique)

</details>

### Exercice 5.2 -- Les deux stratégies face à un problème NP-Complet

**Question :** Complétez le tableau suivant qui résume les deux approches possibles pour traiter un problème d'optimisation NP-Complet en pratique.

In [ ]:
# Exercice 5.2 : Complétez le tableau

tableau = """
| Critère             | Approche exacte              | Approche heuristique         |
|---------------------|------------------------------|------------------------------|
| Optimalité          | ...                          | ...                          |
| Complexité          | ...                          | ...                          |
| Taille max instance | ...                          | ...                          |
| Exemples            | ...                          | ...                          |
| Quand l'utiliser    | ...                          | ...                          |
"""
print(tableau)

# Illustration : comparaison force brute vs heuristique du plus proche voisin

def tsp_force_brute(n, dist_matrix):
    """Résout le TSP par force brute. Retourne (coût, temps)."""
    sommets = list(range(n))
    meilleur = float('inf')
    debut = time.time()
    for perm in itertools.permutations(sommets[1:]):
        cycle = [0] + list(perm)
        cout = sum(dist_matrix[cycle[i]][cycle[(i+1) % n]] for i in range(n))
        meilleur = min(meilleur, cout)
    return meilleur, time.time() - debut

def tsp_plus_proche_voisin(n, dist_matrix):
    """Heuristique du plus proche voisin. Retourne (coût, temps)."""
    debut = time.time()
    visite = [False] * n
    visite[0] = True
    cycle = [0]
    
    for _ in range(n - 1):
        courant = cycle[-1]
        meilleur_voisin = -1
        meilleur_dist = float('inf')
        for j in range(n):
            if not visite[j] and dist_matrix[courant][j] < meilleur_dist:
                meilleur_dist = dist_matrix[courant][j]
                meilleur_voisin = j
        cycle.append(meilleur_voisin)
        visite[meilleur_voisin] = True
    
    cout = sum(dist_matrix[cycle[i]][cycle[(i+1) % n]] for i in range(n))
    return cout, time.time() - debut

# Générer une instance aléatoire
random.seed(42)
n_test = 10
dist_matrix = [[0]*n_test for _ in range(n_test)]
for i in range(n_test):
    for j in range(i+1, n_test):
        d = random.randint(10, 100)
        dist_matrix[i][j] = d
        dist_matrix[j][i] = d

print(f"\n=== Comparaison sur une instance de {n_test} villes ===\n")

cout_exact, temps_exact = tsp_force_brute(n_test, dist_matrix)
cout_heuristique, temps_heuristique = tsp_plus_proche_voisin(n_test, dist_matrix)

ecart = (cout_heuristique - cout_exact) / cout_exact * 100

print(f"{'Méthode':<25} {'Coût':>10} {'Temps':>15} {'Optimalité':>12}")
print("-" * 65)
print(f"{'Force brute (exacte)':<25} {cout_exact:>10} {temps_exact:>14.4f}s {'optimale':>12}")
print(f"{'Plus proche voisin':<25} {cout_heuristique:>10} {temps_heuristique:>14.6f}s {f'+{ecart:.1f}%':>12}")
print(f"\n=> L'heuristique est ~{temps_exact/max(temps_heuristique, 1e-10):.0f}x plus rapide, mais {ecart:.1f}% sous-optimale.")

<details>
<summary><b>Correction Exercice 5.2</b> (cliquez pour révéler)</summary>

| Critère             | Approche exacte                          | Approche heuristique                      |
|---------------------|------------------------------------------|-------------------------------------------|
| Optimalité          | **Garantie** : trouve la meilleure solution | **Aucune garantie** : solution sous-optimale |
| Complexité          | **Exponentielle** : $O(n!)$ ou $O(n^2 \cdot 2^n)$ | **Polynomiale** : ex. $O(n^2)$ ou $O(n^3)$ |
| Taille max instance | Petite ($n \leq 20$ environ)             | Grande ($n$ en milliers ou plus)          |
| Exemples            | Force brute, Held-Karp, Branch & Bound   | Plus proche voisin, 2-opt, recuit simulé  |
| Quand l'utiliser    | Quand $n$ est petit et l'optimalité cruciale | Quand $n$ est grand et une bonne solution suffit |

**Pour notre projet ADEME :** Les tournées de livraison impliquent potentiellement des dizaines ou centaines de villes. Il faudra donc utiliser une **approche heuristique** (sujet des prochains prosits !).

</details>

---

## Partie 6 : Exercices de synthèse

### Exercice 6.1 -- Vrai ou faux ?

Pour chaque affirmation, indiquez si elle est vraie ou fausse et justifiez.

1. Si un problème d'optimisation admet un nombre exponentiel de solutions, le problème de décision associé est nécessairement NP-Complet.
2. $P \subseteq NP$
3. NP signifie "Non Polynomial".
4. Si un problème est NP-Complet et qu'on trouve un algorithme polynomial pour le résoudre, alors $P = NP$.
5. Le problème SAT est le premier problème prouvé NP-Complet (théorème de Cook).
6. La complexité spatiale du TSP est un obstacle majeur à sa résolution.

<details>
<summary><b>Correction Exercice 6.1</b> (cliquez pour reveler)</summary>

1. **FAUX** -- Contre-exemple : le circuit eulerien a un nombre exponentiel de solutions possibles, mais on peut trouver la solution optimale en temps polynomial (algorithme de Hierholzer). Un nombre exponentiel de candidats n'implique pas que le probleme soit NP-Complet.

2. **VRAI** -- Tout probleme resoluble en temps polynomial (classe P) est aussi verifiable en temps polynomial (classe NP). La solution trouvee par l'algorithme sert de certificat.

3. **FAUX** -- NP signifie **"Non-deterministe Polynomial"**. Cela fait reference aux machines de Turing non deterministes, pas a une negation de "polynomial".

4. **VRAI** -- Par definition, tout probleme de NP se reduit polynomialement a un probleme NP-Complet. Si on resout un seul probleme NP-Complet en temps polynomial, alors **tous** les problemes de NP sont dans P, donc $P = NP$.

5. **VRAI** -- Le theoreme de Cook (1971) prouve que le probleme de satisfiabilite booleenne (SAT) est NP-Complet. Tous les autres resultats de NP-Completude utilisent des reductions a partir de SAT (ou de problemes deja prouves NP-Complets via SAT).

6. **FAUX** -- Comme le dit Agathe dans le prosit, la complexite spatiale n'est pas un obstacle majeur avec les capacites des calculateurs modernes. C'est la **complexite temporelle** qui est le veritable probleme.

</details>

### Exercice 6.2 -- Reduction et interpretation

Le schema ci-dessous resume la chaine de reductions menant a la NP-Completude du TSP metrique :

$$\text{SAT} \leq_p \text{3-SAT} \leq_p \text{Vertex Cover} \leq_p \text{Cycle Hamiltonien} \leq_p \text{TSP} \leq_p \text{}\Delta\text{-TSP}$$

**Questions :**

1. Que signifie cette chaine en termes de difficulte relative ?
2. Pourquoi suffit-il de reduire le cycle hamiltonien (et non SAT directement) vers le TSP ?
3. Si demain quelqu'un trouve un algorithme polynomial pour le $\Delta$-TSP, quelles seraient les consequences pour les autres problemes de cette chaine ?

<details>
<summary><b>Correction Exercice 6.2</b> (cliquez pour reveler)</summary>

1. **Difficulte relative :** Chaque probleme a droite est **au moins aussi difficile** que celui a sa gauche. Le $\Delta$-TSP est au moins aussi difficile que tous les problemes de la chaine. Mais comme ils sont **tous NP-Complets**, ils sont en fait **tous equivalents en difficulte** (a une transformation polynomiale pres).

2. **Transitivite de la reduction :** La reduction est **transitive**. Si $A \leq_p B$ et $B \leq_p C$, alors $A \leq_p C$. Comme le cycle hamiltonien est deja prouve NP-Complet (via les reductions precedentes depuis SAT), il suffit de reduire depuis lui. Cela prouve que tous les problemes de NP se reduisent aussi au TSP (par transitivite).

3. **Consequences :** Si on trouve un algorithme polynomial pour le $\Delta$-TSP, alors par transitivite :
   - Le cycle hamiltonien serait dans P
   - Le Vertex Cover serait dans P
   - SAT serait dans P
   - **Tous les problemes de NP** seraient dans P
   - Donc **P = NP** !
   
   C'est precisement pour cela que la resolution efficace d'un seul probleme NP-Complet aurait des consequences enormes.

</details>

### Schema recapitulatif

```
Probleme concret (tournees ADEME)
        |
        v
Modelisation -> TSP metrique (Delta-TSP)
        |
        v
Probleme de decision -> "Existe-t-il un cycle de cout <= k ?"
        |
        v
Appartenance a NP -> Certificat en O(n) ✓
        |
        v
NP-Completude -> Reduction depuis cycle hamiltonien ✓
        |
        v
Conclusion -> PAS d'algorithme polynomial optimal
        |
        v
Suite -> Heuristiques et metaheuristiques (prochains prosits)
```